In [1]:
import numpy as np 
import pandas as pd 
import scipy.stats as stats

In [2]:
df = pd.read_parquet('../data/Divar_cleaned.parquet')
city = pd.read_csv("../data/iran_city_classification.csv")

In [3]:
df_merged = df.merge(city, left_on='city_slug', right_on='نام شهر', how='inner')
df_clean = df_merged.dropna(subset=['building_size', 'دسته‌بندی'])

stats = df_clean.groupby('دسته‌بندی')['building_size'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75)
).round(2)

stats.index.name = 'category'

print(stats)

           count    mean  median     std    Q1     Q3
category                                             
شهر کوچک  469222  212.49   105.0  741.04  78.0  170.0
کلان‌شهر  446877  160.42   100.0  487.66  70.0  145.0


<div dir="rtl">

###  بارگذاری، ادغام و آمار توصیفی اولیه

در این بخش، داده‌های املاک دیوار با فایل دسته‌بندی شهرها ادغام شده و پس از حذف مقادیر گمشده متغیرهای اصلی، آمار توصیفی متراژ (`building_size`) بررسی گردید.

### یافته‌های اولیه:
* **حجم داده ادغام‌شده:** ۹۶۳,۲۷۶ سطر  

* **تعداد رکوردهای معتبر:** ۹۱۶,۰۹۹ سطر (توزیع متوازن بین کلان‌شهرها و شهرهای کوچک)

* **میانگین متراژ:** در شهرهای کوچک ۲۱۲.۵ متر و در کلان‌شهرها ۱۶۰.۴ متر است.

* **میانه متراژ:** در شهرهای کوچک ۱۰۵ متر و در کلان‌شهرها ۱۰۰ متر است.

> **نکته:** اختلاف قابل توجه میانگین و میانه، به همراه انحراف معیار بالا (۷۴۱ در شهرهای کوچک و ۴۸۷ در کلان‌شهرها)، نشان‌‌دهنده وجود **داده‌های پرت شدید (Outliers)** در متغیر متراژ است که پیش از اجرای آزمون فرض باید مدیریت شوند.
</div>

In [ ]:

# ۱. پاکسازی داده‌های پرت (مثلاً متراژ منطقی بین ۲۰ تا ۶۰۰ متر)
df_filtered = df_clean[(df_clean['building_size'] >= 20) & (df_clean['building_size'] <= 600)]

metro = df_filtered[df_filtered['دسته‌بندی'] == 'کلان‌شهر']['building_size']
small = df_filtered[df_filtered['دسته‌بندی'] == 'شهر کوچک']['building_size']

print(f"تعداد نمونه کلان‌شهر پس از فیلتر: {len(metro):,}")
print(f"تعداد نمونه شهر کوچک پس از فیلتر: {len(small):,}")
print(f"میانگین کلان‌شهر: {metro.mean():.2f}")
print(f"میانگین شهر کوچک: {small.mean():.2f}")

t_stat, p_val_t = stats.ttest_ind(metro, small, equal_var=False, alternative='less')
print(f"\nWelch t-statistic: {t_stat:.4f}")
print(f"p-value (t-test): {p_val_t:.4e}")

u_stat, p_val_u = stats.mannwhitneyu(metro, small, alternative='less')
print(f"\nMann-Whitney U statistic: {u_stat:.2f}")
print(f"p-value (Mann-Whitney): {p_val_u:.4e}")

تعداد نمونه کلان‌شهر پس از فیلتر: 428,030
تعداد نمونه شهر کوچک پس از فیلتر: 443,090
میانگین کلان‌شهر: 120.39
میانگین شهر کوچک: 136.31

Welch t-statistic: -81.3383
p-value (t-test): 0.0000e+00

Mann-Whitney U statistic: 86310225742.00
p-value (Mann-Whitney): 0.0000e+00


<div dir="rtl">

### مقایسه متراژ ساختمان در کلان‌شهرها و شهرهای کوچک

ابتدا داده‌های پرت متراژ ساختمان با در نظر گرفتن بازه‌ی منطقی **۲۰ تا ۶۰۰ مترمربع** حذف شدند. سپس متراژ ساختمان‌ها در دو گروه **کلان‌شهر** و **شهر کوچک** جدا شد.


## ۲. فرضیه‌ها

**فرض صفر ($H_0$):** متراژ کلان‌شهرها بزرگ‌تر یا مساوی شهرهای کوچک است.

**فرض مقابل ($H_1$):** متراژ کلان‌شهرها کمتر از شهرهای کوچک است (ادعای اصلی).

برای بررسی تفاوت متراژ ساختمان بین این دو گروه، دو آزمون آماری انجام شد:

* **Welch's t-test یک‌طرفه**: برای بررسی اینکه آیا میانگین متراژ ساختمان در کلان‌شهرها کمتر از شهرهای کوچک است یا خیر.:


* **Mann–Whitney U یک‌طرفه**: یک آزمون غیرپارامتری که برای بررسی تفاوت توزیع متراژ ساختمان بین دو گروه استفاده شد و به نرمال بودن داده‌ها وابستگی ندارد.

در هر دو آزمون، مقدار **p-value** بررسی می‌شود. اگر p-value از سطح معناداری انتخاب‌شده (معمولاً **0.05**) کمتر باشد، شواهد آماری کافی برای رد فرض صفر وجود دارد.
</div>

<div dir="rtl">

# راهنمای آزمون‌های آماری استفاده‌شده و دلیل انتخاب آن‌ها

برای بررسی فرضیه **«کمتر بودن متراژ خانه‌ها در کلان‌شهرها نسبت به شهرهای کوچک»**، از دو آزمون مکمل استفاده شده است. در ادامه دلیل انتخاب و ویژگی‌های هر آزمون آمده است:

## ۱. آزمون Welch's t-test (پارامتری)

### 🔹 چرا این آزمون را انتخاب کردیم؟
در آزمون t-test معمولی (Student's t-test) فرض می‌شود که واریانس (پراکندگی) دو گروه با هم **برابر** است. اما در داده‌های دنیای واقعی (مانند متراژ مسکن):
1. پراکندگی و واریانس در شهرهای کوچک و کلان‌شهرها یکسان نیست.
2. تعداد نمونه‌های دو گروه دقیقاً برابر نیست.

**آزمون Welch** نسخه اصلاح‌شده t-test است که نیازی به یکسان بودن واریانس‌ها ندارد و عملکرد بسیار دقیق‌تری روی داده‌های واقعی ارائه می‌دهد.

### 🔹 چرا نیازی به نرمال‌سازی داده‌ها نبود؟
بر اساس **قضیه حد مرکزی (Central Limit Theorem - CLT)**، اگر حجم نمونه در هر گروه بزرگتر از ۳۰ باشد (در داده‌های ما بالای ۴۰۰,۰۰۰ نمونه است)، توزیع میانگین‌ها خودبه‌خود شکل **نرمال** به خود می‌گیرد. بنابراین آزمون Welch در این حجم داده نسبت به عدم نرمالیّت بسیار **مقاوم (Robust)** عمل می‌کند.


## ۲. آزمون Mann-Whitney U (ناپارامتری)

### 🔹 چرا این آزمون را انتخاب کردیم؟
متغیر متراژ مسکن دارای **چولگی (Skewness)** است؛ یعنی اکثر خانه‌ها متراژ معمولی (مثلاً ۷۰ تا ۱۲۰ متر) دارند اما تعداد محدودی خانه‌ بسیار بزرگ وجود دارد. 

آزمون **Mann-Whitney U** یک آزمون ناپارامتری است؛ یعنی:
* به‌جای **میانگین**، بر اساس **رتبه‌بندی (Ranks) و میانه (Median)** مقایسه را انجام می‌دهد.
* نسبت به داده‌های کج، چوله و حتی داده‌های پرت بسیار مقاوم است.

</div>